# 02 — Data Cleaning
**Project:** HSR Decay in the UPL  
**Purpose:** Filter, normalise, and prepare analysis-ready dataset  
**Input:** Raw merged Catapult CSV  
**Output:** `data/processed/upl_hsr_clean.csv`

In [24]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

# ── CONFIG ──────────────────────────────────────────────────────────────────
RAW_FILE              = '../data/raw/raw_season_physical_data.csv'
OUTPUT_FILE           = '../data/processed/upl_hsr_clean.csv'
MIN_HALF_DURATION_MINUTES = 35                 # Minimum half duration in minutes to include in analysis (substitution threshold)

UPL_CLUBS = [
    'Soltilo Bright Stars FC', 'BUL FC', 'Express FC', 'KCCA FC',
    'Kitara FC', 'Lugazi FC', 'Maroons FC', 'Mbale Heroes FC',
    'Mbarara City FC', 'NEC FC', 'Police FC', 'SC Villa',
    'UPDF FC', 'URA FC', 'Vipers SC', 'Wakiso Giants FC'
]

# ────────────────────────────────────────────────────────────────────────────

df = pd.read_csv(RAW_FILE, low_memory=False)
print(f'Raw rows: {len(df):,}')

Raw rows: 50,046


In [ ]:
import json
import os

# ── ANONYMIZATION ──────────────────────────────────────────────────────────
# Load club anonymization mapping (local only, NOT committed to GitHub)
MAPPING_FILE = '../data/club_mapping.json'
ANONYMIZE = False
CLUB_ANONYMIZATION = {}

if os.path.exists(MAPPING_FILE):
    with open(MAPPING_FILE, 'r') as f:
        CLUB_ANONYMIZATION = json.load(f)
    ANONYMIZE = True
    print(f"Club anonymization ENABLED ({len(CLUB_ANONYMIZATION)} clubs)")
else:
    print("Mapping file not found. Proceeding with REAL club names.")
    print(f"  (Expected at: {os.path.abspath(MAPPING_FILE)})")

✓ Club anonymization ENABLED (16 clubs)


## 1. Standardise Column Names

In [26]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r'\s+', '_', regex=True)
    .str.replace(r'[\/()''-]', '', regex=True)
)

# Rename key columns for convenience
df = df.rename(columns={
    'session_title': 'session_title',
    'player_name':   'player_name',
    'split_name':    'split_name',
    'duration':      'duration_s',
})

#strip all text columns of whitespace
text_cols = df.select_dtypes(include='object').columns
df[text_cols] = df[text_cols].apply(lambda x: x.str.strip())


print('Columns standardised.')

Columns standardised.


## 2. Keep Game Sessions Only (exclude training)

In [27]:
before = len(df)
df = df[df['tags'].str.upper().isin(['GAME', 'GAME TRAINING'])]
print(f'Tags filter: {before:,} → {len(df):,} rows (removed {before - len(df):,})')

Tags filter: 50,046 → 33,815 rows (removed 16,231)


## 3. Keep Half Splits Only

In [28]:
before = len(df)

# Normalise split name casing first
df['split_name'] = df['split_name'].str.strip().str.lower()
df = df[df['split_name'].isin(['1st.half', '2nd.half'])].copy()
df['split_name'] = df['split_name'].map({'1st.half': '1st Half', '2nd.half': '2nd Half'})

print(f'Split filter: {before:,} → {len(df):,} rows (removed {before - len(df):,})')
print(df['split_name'].value_counts())

Split filter: 33,815 → 19,778 rows (removed 14,037)
split_name
1st Half    9889
2nd Half    9889
Name: count, dtype: int64


## 4. Exclude Friendlies and Cup Matches

In [29]:
before = len(df)
title_upper = df['session_title'].str.upper()
mask_exclude = title_upper.str.contains('FRIENDLY|CUP', na=False, regex=True)
df = df[~mask_exclude].copy()
print(f'Friendly/Cup filter: {before:,} → {len(df):,} rows (removed {before - len(df):,})')

Friendly/Cup filter: 19,778 → 18,273 rows (removed 1,505)


## 5. Session Title Pattern Filter
Keep only sessions matching the standard UPL format: `MD##-Club-Club-Location-League-Result`

In [30]:
# Clean session title first
df['session_title'] = (
    df['session_title']
    .str.strip()
    .str.replace(r'\.+', '', regex=True)
    .str.replace(r'\s*-\s*', '-', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
)

before = len(df)
pattern = r'^MD\d+-[\w\s]+-[\w\s]+-[\w\s]+-[\w\s]+-[\w\s]+$'
df = df[df['session_title'].str.match(pattern, case=False, na=False)].copy()
df['session_title'] = df['session_title'].str.upper()
print(f'Pattern filter: {before:,} → {len(df):,} rows (removed {before - len(df):,})')

Pattern filter: 18,273 → 12,916 rows (removed 5,357)


## 6. Parse Session Title Components

In [31]:
regex = re.compile(
    r'^(MD\d+)-'
    r'(.+?)-'
    r'(.+?)-'
    r'(.+?)-'
    r'(.+?)-'
    r'(.+?)$',
    re.IGNORECASE
)

extracted = df['session_title'].str.extract(regex)
extracted.columns = ['match_day', 'club_for', 'club_against', 'part1', 'part2', 'part3']

location_set = {'home', 'away'}
league_set   = {'league'}
result_set   = {'win', 'loss', 'draw', 'lose'}

extracted['location'] = None
extracted['league']   = None
extracted['result']   = None

for i, row in extracted.iterrows():
    for val in [row['part1'], row['part2'], row['part3']]:
        if pd.isna(val):
            continue
        v = val.strip().lower()
        if v in location_set:
            extracted.at[i, 'location'] = v.title()
        elif v in league_set:
            extracted.at[i, 'league'] = v.title()
        elif v in result_set:
            extracted.at[i, 'result'] = 'Loss' if v == 'lose' else v.title()

valid = extracted.dropna(subset=['location', 'league', 'result']).copy()
df = df.loc[valid.index].reset_index(drop=True)
valid = valid.drop(columns=['part1', 'part2', 'part3']).reset_index(drop=True)
df = pd.concat([df, valid[['match_day', 'club_for', 'club_against', 'location', 'result']]], axis=1)

print(f'Rows after session parsing: {len(df):,}')
print('\nResult distribution:')
print(df.drop_duplicates('session_title')['result'].value_counts())

Rows after session parsing: 12,916

Result distribution:
result
Win     159
Loss    138
Draw    119
Name: count, dtype: int64


## 7. Club Name Normalisation + Whitelist Filter

In [32]:
CLUB_ALIASES = {
    r'mbarara(?!\s+city)': 'Mbarara City FC',
    r'bright\s+stars':     'Soltilo Bright Stars FC',
    r'^soltilo$':          'Soltilo Bright Stars FC',
    r'^vipers\s+fc$':      'Vipers SC',
    r'sc\s+villa': 'SC Villa',
}

def normalise_club(name):
    if pd.isna(name):
        return name
    n = name.strip()
    for pattern, replacement in CLUB_ALIASES.items():
        if re.search(pattern, n, re.IGNORECASE):
            return replacement
    return n

df['club_for']     = df['club_for'].apply(normalise_club)
df['club_against'] = df['club_against'].apply(normalise_club)

CANONICAL = {c.upper().replace(' FC','').replace(' SC','').strip(): c for c in UPL_CLUBS}

def to_canonical(name):
    if pd.isna(name):
        return name
    key = re.sub(r'\b(FC|SC)\b', '', str(name), flags=re.IGNORECASE).strip().upper()
    return CANONICAL.get(key, name)

df['club_for']     = df['club_for'].apply(to_canonical)
df['club_against'] = df['club_against'].apply(to_canonical)

# Whitelist: club_for must fuzzy-match one of the 16 UPL clubs
def match_club(name, whitelist):
    if pd.isna(name):
        return False
    name_clean = re.sub(r'\b(FC|SC)\b', '', str(name), flags=re.IGNORECASE).strip().upper()
    for club in whitelist:
        club_clean = re.sub(r'\b(FC|SC)\b', '', club, flags=re.IGNORECASE).strip().upper()
        if club_clean in name_clean or name_clean in club_clean:
            return True
    return False

before = len(df)
df = df[df['club_for'].apply(lambda x: match_club(x, UPL_CLUBS))].copy()
print(f'Whitelist filter: {before:,} → {len(df):,} rows (removed {before - len(df):,})')

print('\nClubs remaining after whitelist:')
print(sorted(df['club_for'].unique()))

Whitelist filter: 12,916 → 12,890 rows (removed 26)

Clubs remaining after whitelist:
['BUL FC', 'Express FC', 'KCCA FC', 'Kitara FC', 'Lugazi FC', 'Maroons FC', 'Mbale Heroes FC', 'Mbarara City FC', 'NEC FC', 'Police FC', 'SC Villa', 'Soltilo Bright Stars FC', 'UPDF FC', 'URA FC', 'Vipers SC', 'Wakiso Giants FC']


## 8. Parse Position and Exclude Goalkeepers

In [33]:
POSITION_MAP = {
    'GK': 'GK',
    'CB': 'DEF', 'RB': 'DEF', 'LB': 'DEF', 'DF': 'DEF',
    'RWB': 'DEF', 'LWB': 'DEF', 'SW': 'DEF', 'CD': 'DEF',
    'DM': 'MID', 'CM': 'MID', 'AM': 'MID', 'AMC': 'MID',
    'DMC': 'MID', 'RM': 'MID', 'LM': 'MID', 'MF': 'MID',
    'CAM': 'MID', 'CDM': 'MID', 'MC': 'MID',
    'ST': 'FWD', 'CF': 'FWD', 'LW': 'FWD', 'RW': 'FWD',
    'SS': 'FWD', 'FW': 'FWD', 'FWD':'FWD','LCB':'DEF','MD':'MID','DC':'DEF'
}

df['position_raw'] = df['player_name'].str.strip().str.split('-').str[-1].str.strip().str.upper()
df['position'] = df['position_raw'].map(POSITION_MAP)

unmapped = df[df['position'].isna()]['position_raw'].value_counts()
if len(unmapped) > 0:
    print('Still unmapped — add to POSITION_MAP:')
    print(unmapped)
else:
    print('All positions mapped.')

before = len(df)
df = df[df['position'] != 'GK'].copy()
df = df[df['position'] != 'REFEREE'].copy()

print(f'\nGK & Referee exclusion: {before:,} → {len(df):,} rows')
print('\nPosition distribution:')
print(df['position'].value_counts())

Still unmapped — add to POSITION_MAP:
position_raw
REFEREE    2
Name: count, dtype: int64

GK & Referee exclusion: 12,890 → 12,080 rows

Position distribution:
position
DEF    4233
MID    4232
FWD    3613
Name: count, dtype: int64


## 9. Duration Filter (substitute threshold)

In [34]:
df['duration_s'] = pd.to_numeric(df['duration_s'], errors='coerce')

before = len(df)
df = df[df['duration_s'] >= MIN_HALF_DURATION_MINUTES * 60].copy()
print(f'Duration filter (>= {MIN_HALF_DURATION_MINUTES} minutes): {before:,} → {len(df):,} rows (removed {before - len(df):,})')

Duration filter (>= 35 minutes): 12,080 → 10,645 rows (removed 1,435)


## 10. Calculate HSR Rate

In [35]:
# Identify SZ4 and SZ5 columns (handle minor naming variations)
sz4_col = [c for c in df.columns if 'speed_zone_4' in c and 'km' in c][0]
sz5_col = [c for c in df.columns if 'speed_zone_5' in c and 'km' in c][0]
print(f'Using: {sz4_col}')
print(f'Using: {sz5_col}')

df[sz4_col] = pd.to_numeric(df[sz4_col], errors='coerce')
df[sz5_col] = pd.to_numeric(df[sz5_col], errors='coerce')

df['hsr_m']    = (df[sz4_col] + df[sz5_col]) * 1000
df['hsr_rate'] = df['hsr_m'] / (df['duration_s'] / 60)

print(f'\nHSR Rate summary:')
print(df.groupby('split_name')['hsr_rate'].describe().round(2))

Using: distance_in_speed_zone_4_km
Using: distance_in_speed_zone_5_km

HSR Rate summary:
             count  mean   std  min  25%   50%    75%    max
split_name                                                  
1st Half    5432.0  7.75  6.09  0.0  0.0  8.34  12.28  31.74
2nd Half    5213.0  6.96  4.89  0.0  3.4  6.79  10.19  27.36


## 11. Require Both Halves Per Player-Match
Drop any player-match where only one half survived all filters

In [36]:
df['player_match_id'] = df['player_name'].str.strip() + ' | ' + df['session_title'].str.strip()

before = len(df)
pair_counts = df.groupby('player_match_id')['split_name'].nunique()
valid_ids = pair_counts[pair_counts == 2].index
df = df[df['player_match_id'].isin(valid_ids)].copy()

print(f'Paired filter: {before:,} → {len(df):,} rows')
print(f'Valid player-match pairs: {len(valid_ids):,}')

Paired filter: 10,645 → 9,990 rows
Valid player-match pairs: 4,992


## 12. Final Dataset Summary

In [39]:
print('=' * 55)
print('CLEAN DATASET SUMMARY')
print('=' * 55)
print(f'Total rows:                   {len(df):>8,}')
print(f'Unique clubs:                 {df["club_for"].nunique():>8}')
print(f'Unique matches:               {df["session_title"].nunique():>8}')
print(f'Unique players:               {df["player_name"].nunique():>8}')
print(f'Valid player-match pairs:     {len(valid_ids):>8,}')
print('='*55)
print('\nPosition breakdown:')
print(df[df['split_name']=='1st Half']['position'].value_counts())
print('\nResult breakdown:')
print(df[df['split_name']=='1st Half']['result'].value_counts())


CLEAN DATASET SUMMARY
Total rows:                      9,990
Unique clubs:                       16
Unique matches:                    412
Unique players:                    405
Valid player-match pairs:        4,992

Position breakdown:
position
DEF    1923
MID    1684
FWD    1388
Name: count, dtype: int64

Result breakdown:
result
Win     1964
Loss    1671
Draw    1361
Name: count, dtype: int64


## 13. Save Clean Dataset

In [40]:
keep_cols = [
    'player_match_id', 'session_title', 'match_day', 'player_name',
    'club_for', 'club_against', 'location', 'result',
    'split_name', 'position', 'position_raw',
    'duration_s', 'hsr_m', 'hsr_rate'
]

df_out = df[keep_cols].reset_index(drop=True)

# Apply club anonymization if mapping is available
if ANONYMIZE:
    df_out['club_for'] = df_out['club_for'].map(CLUB_ANONYMIZATION)
    df_out['club_against'] = df_out['club_against'].map(CLUB_ANONYMIZATION)
    print("Club names anonymized")

df_out.to_csv(OUTPUT_FILE, index=False)
print(f'Saved: {OUTPUT_FILE}')
print(f'Shape: {df_out.shape}')

Club names anonymized
Saved: ../data/processed/upl_hsr_clean.csv
Shape: (9990, 14)
